# Fase 1 — Extracción de datos

**TFM · Modelo predictivo 1X2 de LaLiga**

Este notebook documenta la **primera fase** del pipeline: la *extracción* de datos crudos
desde las distintas fuentes. No entra en el detalle de *cómo* se hace cada scraping (eso vive
en `scripts/extractors/*.py`, código productivo y testeado), pero sí deja claro **cómo está
montada la extracción para que sea automática**, **qué variables se obtienen de cada fuente** 
y da un **primer vistazo** a la forma de cada tabla.

El resultado son varios ficheros `.parquet` en `data/02_processed/`. En el notebook siguiente
(**`02_CONSOLIDACION_MASTER.ipynb`**) todos se unen en una única tabla maestra por partido.

---


## 1 · Cómo está montada la extracción (automática y reproducible)

Cada fuente tiene su extractor en `scripts/extractors/`: `footballdata.py`, `understat.py`,
`sofascore.py`, `clubelo.py`, `transfermarkt.py`, `fatigue.py`, `availability.py`,
`lineups.py` y `understat_shots.py`, con `teamnames.py` como capa compartida; `fbref.py`
queda como comprobación cruzada. Todos
comparten el **mismo contrato**, pensado para ejecutarse cada jornada sin intervención:

**a) Dos modos: descarga histórica (*backfill*) y actualización incremental (*update*).**
- El **backfill** se ejecuta *una vez*: recorre todo el histórico de la ventana del proyecto
  (temporada 2016/17 en adelante) y construye la tabla completa desde cero.
- El **update** es lo que se lanza *cada jornada*: detecta qué partidos/jornadas son nuevos y
  descarga **solo eso**, añadiéndolo a lo que ya había. Es la operación normal del pipeline en
  producción.

**b) Idempotencia — re-ejecutar nunca duplica ni re-descarga de más.**
Cada fila lleva una **clave estable** (p.ej. `match_key`, o `año|club|jugador` en Transfermarkt).
Antes de descargar, el extractor mira qué claves ya tiene y **salta** lo existente. Esto hace que:
- volver a lanzar un update sea seguro (no crea duplicados),
- si una descarga falla a medias, al re-lanzarla **solo reintente lo que faltó**.

**c) Tolerancia a fallos — checkpoints + guardado atómico.**
- Durante descargas largas se guardan **checkpoints** cada N filas, así un corte de red no
  tira todo el trabajo ya hecho.
- El guardado es **atómico** (se escribe a un temporal y se renombra al final): un fallo a
  mitad **nunca** deja un `.parquet`/`.csv` corrupto o a medio escribir.

**d) Dos capas de datos — crudo y procesado.**
- `data/01_raw/<fuente>/…` → **verdad cruda** tal cual se descargó (CSV). No se toca.
- `data/02_processed/<fuente>.parquet` → versión **tipada, limpia y con claves canónicas**,
  lista para cruzar. Si el CSV crudo se edita a mano, basta reprocesar para regenerar el parquet.

**e) Nombres de equipo unificados (`teamnames.py`).**
Cada fuente nombra a los clubes distinto (`"Athletic Club"`, `"Bilbao"`, `"ath-bilbao"`).
`teamnames.py` reduce cualquier variante a una **clave canónica** estable (`athletic_bilbao`)
y construye el **`match_key`** `AAAA-MM-DD|local|visitante`, que es el pegamento entre fuentes.
Si aparece un nombre nuevo sin mapear, el extractor **falla ruidosamente** en vez de publicar
un cruce con huecos silenciosos.

**f) Ejecución.** Cada extractor expone una CLI (`diag` / `backfill` / `update` / `process`).
Los scrapers en vivo se lanzan en local; el reprocesado y la orquestación de notebooks
(vía *papermill*) generan las tablas que consumen las fases siguientes.


### Las cuatro familias de señal

| Familia | Pregunta que responde | Fuentes |
|---|---|---|
| **Resultado + mercado** | ¿Qué pasó y qué esperaban las casas? | football-data.co.uk |
| **Calidad de juego** | ¿Mereció ganar quien ganó, y cómo jugó? | Understat, Sofascore |
| **Fuerza / forma** | ¿Cómo de bueno es cada equipo ahora? | ClubElo, Transfermarkt |
| **Estado del equipo** | ¿Llega entero, descansado y sin bajas? | Fatiga, Disponibilidad, Alineaciones |

---


## 2 · Preparación

Localizamos la carpeta de datos y definimos ayudantes: `vistazo()` (forma + muestra),
`variables()` (colapsa las columnas `*_home`/`*_away` en su **variable única** para poder
listar todo sin repetir cada métrica dos veces) y `familias_sofa()` (agrupa las  124 columnas
de Sofascore por categoría).


In [11]:
import os, re
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 60); pd.set_option("display.width", 200)

def _find_data_dir(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        d = cand / "data" / "02_processed"
        if d.is_dir():
            return d
    raise FileNotFoundError("No encuentro data/02_processed/ desde " + str(p))
DATA = _find_data_dir()
print("Datos procesados en:", DATA)

def variables(cols):
    """Colapsa columnas gemelas local/visitante en su variable unica (mantiene el orden)."""
    out = []
    for c in cols:
        b = re.sub(r"^us_(home|away)_", "us_", c)
        b = re.sub(r"^(home|away)_", "", b)
        b = re.sub(r"_(home|away|draw)$", "", b)
        if re.match(r"^(odds_|mkt_p|ah_)", b):
            b = re.sub(r"_(h|d|a)$", "", b)
        if b not in out:
            out.append(b)
    return out

def vistazo(fichero, cols=None, n=5, titulo=None, inventario=True, opcional=None):
    """Muestra un artefacto procesado. `opcional` = orden que lo genera.

    Dos de los volcados del proyecto -las alineaciones y el detalle de tiro-
    piden una peticion por partido y se reconstruyen una sola vez. Toda la
    cadena aguas abajo los trata como OPCIONALES: si el parquet no esta, el
    master se construye igual y el bloque queda en hueco declarado. Este
    ayudante sigue el mismo criterio, de modo que el notebook se recorre
    entero antes de haber lanzado esos volcados y la ausencia se lee como lo
    que es -un bloque todavia no extraido- y no como una traza de error.
    Cualquier otro artefacto sigue siendo exigible: si falta, el fallo es real
    y debe cortar.
    """
    ruta = DATA / fichero
    if not ruta.exists():
        if opcional is None:
            raise FileNotFoundError(
                f"Falta {fichero} en {DATA}. Es una fuente exigible: reprocesa "
                "su extractor antes de seguir.")
        print("[ausente] " + (titulo or fichero))
        print(f"   `{fichero}` no esta en data/02_processed/. Es uno de los dos "
              "volcados largos,")
        print("   declarados opcionales en toda la cadena. Para generarlo:")
        print("      " + opcional)
        return None
    df = pd.read_parquet(ruta)
    print((titulo or fichero) + "  ->  {:,} filas x {} columnas".format(len(df), df.shape[1]))
    if inventario:
        v = variables(df.columns)
        print("Variables unicas ({}):".format(len(v)))
        print("   " + ", ".join(v))
    display((df[cols] if cols else df).head(n))
    return df

_GRUPOS = ["match_overview","shots","attack","passes","duels","defending","goalkeeping"]
def familias_sofa(df):
    from collections import OrderedDict
    fam = OrderedDict((g, []) for g in _GRUPOS)
    for c in variables(df.columns):
        if not c.startswith("sofa_"): continue
        rest = c[len("sofa_"):]
        for g in _GRUPOS:
            if rest.startswith(g + "_"):
                fam[g].append(rest[len(g)+1:]); break
    for g, ms in fam.items():
        print("[{}]  ({}): {}".format(g, len(ms), ", ".join(ms)))


Datos procesados en: C:\Users\admin\Desktop\TFM 2026\TFM\repos\laliga-predictor-engine\data\02_processed


## 3 · Resumen de fuentes

| Fichero (`data/02_processed/`) | Fuente | Granularidad | Nº variables | Cobertura |
|---|---|---|---|---|
| `footballdata_match.parquet` | football-data.co.uk | Partido | 50 col (resultado + cuotas) | 2016/17– |
| `understat_match.parquet` | Understat | Partido | xG, npxG, PPDA, deep, xPts (×2 lados) | 2016/17– |
| `sofascore_team_match.parquet` | Sofascore | Partido |  124 métricas de juego | 2016/17– |
| `clubelo_elo.parquet` | ClubElo | Club / intervalo | Elo, rank, level | 2016/17– |
| `team_squad_value.parquet` | Transfermarkt | Club-temporada | valor, tamaño, edad | 2016/17– |
| `transfermarkt_player_values.parquet` | Transfermarkt | Jugador-temporada | valor por jugador | 2016– |
| `team_fatigue.parquet` | Sofascore (calendario) | Evento / equipo | descanso, congestión, viajes | 2016– |
| `team_availability.parquet` | Sofascore | Evento / equipo | bajas (lesión/sanción/duda) | 2016– |
| `team_lineups.parquet` | Sofascore | Partido / equipo | rotación del once y carga de minutos | 2016/17– |
| `player_minutes.parquet` | Sofascore | Jugador-partido | minutos y titularidad | 2016– |
| `team_shot_quality.parquet` | Understat (tiros) | Partido / equipo | forma de la distribución de remates | 2016/17– |
| `understat_shots.parquet` | Understat (tiros) | Disparo | detalle de cada remate (descriptivo) | 2016/17– |

**Ventana temporal única.** Todas las fuentes se recortan a la **temporada 2016/17 en
adelante**, que es la ventana común donde conviven las tablas finas por partido (Sofascore,
Understat, fatiga, disponibilidad). Aunque algunas fuentes (football-data, ClubElo,
Transfermarkt) tienen histórico más largo en su capa cruda, la capa procesada arranca toda
en 2016/17 para que el máster sea homogéneo: no tiene sentido mezclar unas variables desde
2016 con otras desde más atrás. En ClubElo se conserva además el tramo de Elo *vigente* al
inicio de la 16/17 (su `from` puede ser algo anterior) para que el cruce a fecha del partido
encuentre siempre un rating válido.

---


## 4 · football-data.co.uk — resultado y mercado

Una fila por partido. Es la **columna vertebral** del dataset y la fuente de la **variable
objetivo**. Se extrae:

- **Resultado** — a tiempo completo `fthg`/`ftag` (goles local/visitante) y **`ftr`** (`H`/`D`/`A`,
  el *target 1X2*); y al descanso `hthg`/`htag`/`htr`.
- **Cuotas por casa** — Bet365 (`odds_b365_*`) y Pinnacle (`odds_ps_*`), en formato `_h/_d/_a`.
- **Agregados de mercado (apertura)** — máximo y media de todas las casas: `odds_max_*`, `odds_avg_*`.
- **Cuotas de cierre** — `odds_close_avg_*`, `odds_close_max_*`, `odds_b365_close_*`
  (la línea justo antes del partido, la más informada).
- **Más de 2.5 / Menos de 2.5 goles** — `odds_over25`, `odds_under25` (+ variantes Bet365 y cierre).
- **Hándicap asiático** — línea y cuotas: `ah_line`, `odds_ah_home/away` (+ cierre).
- **Probabilidad implícita del mercado** — `mkt_p_home/draw/away` (cuotas convertidas a
  probabilidad y normalizadas: el consenso del mercado, listo para usar como *feature* y *benchmark*).


In [15]:
fd = vistazo("footballdata_match.parquet",
    cols=["match_key","date","home_team","away_team","fthg","ftag","ftr",
          "odds_b365_h","odds_b365_d","odds_b365_a","mkt_p_home","mkt_p_draw","mkt_p_away"],
    titulo="football-data (resultado + mercado)")
print("\nRango de fechas:", fd["date"].min(), "->", fd["date"].max())


football-data (resultado + mercado)  ->  3,800 filas x 50 columnas
Variables unicas (30):
   match_key, season, season_year, date, team, key, fthg, ftag, ftr, hthg, htag, htr, odds_b365, odds_ps, odds_max, odds_avg, odds_close_avg, odds_close_max, odds_b365_close, odds_over25, odds_under25, odds_b365_over25, odds_b365_under25, odds_close_over25, odds_close_under25, ah_line, odds_ah, ah_close_line, odds_ah_close, mkt_p


,match_key,date,home_team,away_team,fthg,ftag,ftr,odds_b365_h,odds_b365_d,odds_b365_a,mkt_p_home,mkt_p_draw,mkt_p_away
0,2016-08-19|deportivo_la_coruna|eibar,2016-08-19,La Coruna,Eibar,2,1,H,1.95,3.4,4.2,0.48392,0.288013,0.228067
1,2016-08-19|malaga|osasuna,2016-08-19,Malaga,Osasuna,1,1,D,1.62,3.75,6.0,0.587462,0.260024,0.152514
2,2016-08-20|barcelona|real_betis,2016-08-20,Barcelona,Betis,6,2,H,1.08,11.0,29.0,0.87317,0.091515,0.035316
3,2016-08-20|granada|villarreal,2016-08-20,Granada,Villarreal,1,1,D,2.5,3.2,3.0,0.372944,0.302645,0.324411
4,2016-08-20|sevilla|espanyol,2016-08-20,Sevilla,Espanol,6,4,H,1.67,4.0,5.0,0.56543,0.246094,0.188477



Rango de fechas: 2016-08-19 -> 2026-05-24


## 5 · Understat — calidad de juego (xG y derivados)

Understat aporta métricas de *modelo* que no están en el marcador, **todas para local y
visitante**:

- **`us_xg`** — goles esperados: calidad total de las ocasiones generadas.
- **`us_npxg`** — xG **sin penaltis** (*non-penalty xG*): mide el juego en abierto, sin el ruido
  de los penaltis.
- **`us_ppda`** — *passes allowed per defensive action*: cuántos pases deja hacer al rival antes
  de una acción defensiva. **Cuanto más bajo, más presión alta** ejerce el equipo.
- **`us_deep`** — pases/acciones completadas cerca del área rival (~últimos 20 m): dominio territorial.
- **`us_xpts`** — puntos esperados: los puntos que "merecía" el equipo dado el xG del partido.


In [16]:
us = vistazo("understat_match.parquet",
    cols=["date","home_team","away_team","us_home_xg","us_away_xg",
          "us_home_npxg","us_home_ppda","us_home_deep","us_home_xpts"],
    titulo="Understat (xG y derivados)")
print("\nRango de fechas:", us["date"].min(), "->", us["date"].max())


Understat (xG y derivados)  ->  3,800 filas x 20 columnas
Variables unicas (13):
   match_id, season, season_year, datetime, date, team, key, match_key, us_xg, us_npxg, us_ppda, us_deep, us_xpts


,date,home_team,away_team,us_home_xg,us_away_xg,us_home_npxg,us_home_ppda,us_home_deep,us_home_xpts
0,2016-08-19,Malaga,Osasuna,0.834224,0.829951,0.834224,5.172414,4.0,1.2471
1,2016-08-20,Deportivo La Coruna,Eibar,1.619700,0.359012,0.876427,16.083333,0.0,2.5060
2,2016-08-20,Barcelona,Real Betis,3.843250,0.250977,3.843250,5.318182,33.0,2.9555
3,2016-08-20,Granada,Villarreal,0.977176,1.758810,0.977176,3.225806,1.0,0.7308
4,2016-08-21,Sevilla,Espanyol,2.602890,1.418030,2.602890,4.120000,16.0,2.1614



Rango de fechas: 2016-08-19 -> 2026-05-24


## 6 · Sofascore — radiografía completa del partido

La fuente más rica:  124 métricas por partido a nivel de equipo (cada una para local y
visitante), organizadas en **siete grupos**. Además del **rating medio** de cada plantilla
(`home_avg_rating`/`away_avg_rating`) y la estimación de xG de Sofascore. Los grupos son:

- **match_overview** — posesión, tiros totales, córners, faltas, pases, tiros libres, tarjetas,
  ocasiones claras, entradas, xG, **distancia recorrida** y **sprints**.
- **shots** — desglose del disparo: a puerta, fuera, al palo, bloqueados, dentro/fuera del área.
- **attack** — fueras de juego, balones en profundidad, ocasiones claras marcadas/falladas,
  faltas recibidas en último tercio, toques en el área.
- **passes** — pases acertados, saques de banda, entradas al último tercio, balones largos, centros.
- **duels** — duelos totales, terrestres, aéreos, pérdidas de balón, regates.
- **defending** — entradas (ganadas/totales), intercepciones, recuperaciones, despejes,
  errores que acaban en tiro/gol.
- **goalkeeping** — paradas, saques de puerta, paradas decisivas, despejes de puños, atrapadas
  altas, **goles evitados**, paradas de penalti.

La celda lista **todas** las métricas agrupadas.


In [17]:
so = vistazo("sofascore_team_match.parquet",
    cols=["match_key","season_name","home_key","away_key","home_avg_rating","away_avg_rating"],
    titulo="Sofascore (estadistica de partido)", inventario=False)
print("Columnas totales:", so.shape[1], "| Rating y grupos de metricas:\n")
familias_sofa(so)


Sofascore (estadistica de partido)  ->  3,800 filas x 124 columnas


,match_key,season_name,home_key,away_key,home_avg_rating,away_avg_rating
0,2016-08-19|malaga|osasuna,LaLiga 16/17,malaga,osasuna,6.735714,6.885714
1,2016-08-19|deportivo_la_coruna|eibar,LaLiga 16/17,deportivo_la_coruna,eibar,6.835714,6.785714
2,2016-08-20|barcelona|real_betis,LaLiga 16/17,barcelona,real_betis,7.392308,6.335714
3,2016-08-20|granada|villarreal,LaLiga 16/17,granada,villarreal,6.723077,6.725000
4,2016-08-20|sevilla|espanyol,LaLiga 16/17,sevilla,espanyol,7.128571,6.450000


Columnas totales: 124 | Rating y grupos de metricas:

[match_overview]  (14): ball_possession, total_shots, goalkeeper_saves, corner_kicks, fouls, passes, free_kicks, yellow_cards, red_cards, big_chances, tackles, expected_goals, distance_covered, number_of_sprints
[shots]  (7): total_shots, shots_on_target, hit_woodwork, shots_off_target, blocked_shots, shots_inside_box, shots_outside_box
[attack]  (6): offsides, through_balls, big_chances_scored, big_chances_missed, fouled_in_final_third, touches_in_penalty_area
[passes]  (6): accurate_passes, throw_ins, final_third_entries, long_balls, crosses, final_third_phase
[duels]  (5): duels, ground_duels, aerial_duels, dispossessed, dribbles
[defending]  (7): tackles_won, total_tackles, interceptions, recoveries, clearances, errors_lead_to_a_shot, errors_lead_to_a_goal
[goalkeeping]  (7): total_saves, goal_kicks, big_saves, punches, high_claims, goals_prevented, penalty_saves


## 7 · ClubElo — rating de fuerza dinámico

Formato largo: una fila por club y **intervalo de validez** del rating (`from` → `to`), no por
día suelto. Columnas: **`team`** (nombre original) y **`team_key`** (normalizado), **`country`**
y **`level`** (país y división), **`rank`** (puesto mundial ClubElo), **`elo`** (el rating), y las
fechas del intervalo **`from`**/**`to`** más **`date`** (inicio del tramo). El Elo sube al ganar
—más si el rival era fuerte— y baja al perder; se **actualiza solo** partido a partido y es
comparable entre equipos. En el máster se toma el Elo vigente de cada equipo *el día del partido*
buscando el intervalo que lo contiene.


In [18]:
ce = vistazo("clubelo_elo.parquet", titulo="ClubElo (rating por intervalos)")
print("\nRango de fechas:", ce["date"].min(), "->", ce["date"].max(),
      "| Clubes distintos:", ce["team_key"].nunique())


ClubElo (rating por intervalos)  ->  32,891 filas x 9 columnas
Variables unicas (9):
   team, team_key, country, level, rank, elo, from, to, date


,team,team_key,country,level,rank,elo,from,to,date
0,Alaves,alaves,ESP,2,90.0,1628.29541,2016-06-19,2016-07-01,2016-06-19
1,Alaves,alaves,ESP,1,90.0,1628.29541,2016-07-02,2016-07-28,2016-07-02
2,Alaves,alaves,ESP,1,91.0,1628.29541,2016-07-29,2016-07-29,2016-07-29
3,Alaves,alaves,ESP,1,90.0,1628.29541,2016-07-30,2016-08-04,2016-07-30
4,Alaves,alaves,ESP,1,92.0,1628.29541,2016-08-05,2016-08-05,2016-08-05



Rango de fechas: 2016-05-19 -> 2026-08-30 | Clubes distintos: 31


## 8 · Transfermarkt — valor de plantilla (club-temporada)

Un *prior* de fuerza independiente de los resultados. Columnas: **`season`**, **`team_key`**,
**`squad_value`** (valor total de la plantilla, M€), **`squad_size`** (nº de jugadores),
**`value_avg_mio`** (valor medio por jugador) y **`avg_age`** (edad media). Útil sobre todo a
principio de temporada, cuando aún hay pocos partidos que alimenten al Elo.


In [20]:
sv = vistazo("team_squad_value.parquet", titulo="Transfermarkt (valor de plantilla)")


Transfermarkt (valor de plantilla)  ->  220 filas x 6 columnas
Variables unicas (6):
   season, team_key, squad_value, squad_size, value_avg_mio, avg_age


,season,team_key,squad_value,squad_size,value_avg_mio,avg_age
0,2016,alaves,85.4,30,2.944828,25.333333
1,2016,athletic_bilbao,172.6,30,5.753333,26.866667
2,2016,atletico_madrid,515.8,38,18.421429,24.657895
3,2016,barcelona,765.3,33,23.190909,25.757576
4,2016,celta_vigo,122.4,29,4.220690,25.448276


## 9 · Transfermarkt — valor por jugador

Una fila por jugador-temporada. Columnas: **`season`**, **`team_key`**, **`player`**,
**`player_key`** (clave normalizada), **`player_id`** (id de Transfermarkt) y **`value_mio`**
(valor de mercado, M€). Permite **ponderar las bajas por valor**: cruzando con disponibilidad
se obtiene *cuánto valor* de la plantilla falta, no solo *cuántos* jugadores.


In [22]:
pv = vistazo("transfermarkt_player_values.parquet", titulo="Transfermarkt (valor por jugador)")
print("\nJugadores distintos:", pv["player_key"].nunique(),
      "| Temporadas:", pv["season"].min(), "->", pv["season"].max())



Transfermarkt (valor por jugador)  ->  7,854 filas x 6 columnas
Variables unicas (6):
   season, team_key, player, player_key, player_id, value_mio


,season,team_key,player,player_key,player_id,value_mio
0,2017,real_madrid,Keylor Navas,keylor-navas,79422,18.0
1,2017,real_madrid,Kiko Casilla,kiko-casilla,27486,6.0
2,2017,real_madrid,Luca Zidane,luca-zidane,283489,0.3
3,2017,real_madrid,Moha Ramos,moha-ramos,495602,NaN
4,2017,real_madrid,Raphaël Varane,raphael-varane,164770,70.0



Jugadores distintos: 2795 | Temporadas: 2016 -> 2026


## 10 · Fatiga y congestión de calendario

Construida desde el calendario **multi-competición** de Sofascore (liga + copa + Europa +
selecciones), una fila por equipo y evento. Se divide en:

- **Identidad** — `row_id`, `event_id`, `team_id`, `team`, `team_key`, `opponent`, `opp_key`,
  `is_home`, `date`, `start_ts`.
- **Competición** — `competition`, `comp_id`, `is_friendly`, `is_competitive`, `status`.
- **Señales de fatiga** — **`rest_days`** (días de descanso desde el partido anterior, contando
  *todas* las competiciones), **`games_last_7`** y **`games_last_14`** (partidos en los últimos
  7/14 días), **`is_midweek`** (juega entre semana), **`euro_trip_recent`** (vuelve de un viaje
  europeo) e **`is_post_intl_break`** (primer partido tras un parón de selecciones).


In [23]:
fa = vistazo("team_fatigue.parquet",
    cols=["date","team_key","opp_key","is_home","competition",
          "rest_days","games_last_7","games_last_14","is_midweek","euro_trip_recent","is_post_intl_break"],
    titulo="Fatiga (descanso y congestion)", inventario=False)
print("Todas las columnas:", list(fa.columns))


Fatiga (descanso y congestion)  ->  15,554 filas x 21 columnas


,date,team_key,opp_key,is_home,competition,rest_days,games_last_7,games_last_14,is_midweek,euro_trip_recent,is_post_intl_break
0,2016-08-20,espanyol,sevilla,False,LaLiga,97.135417,0,0,False,False,False
1,2016-08-26,espanyol,malaga,True,LaLiga,5.989583,1,1,False,False,False
2,2016-09-09,espanyol,real_sociedad,False,LaLiga,13.947917,0,1,False,False,True
3,2016-09-18,espanyol,real_madrid,True,LaLiga,9.000000,0,1,False,False,False
4,2016-09-22,espanyol,osasuna,False,LaLiga,3.968750,1,2,True,False,False


Todas las columnas: ['row_id', 'event_id', 'team_id', 'team', 'team_key', 'opponent', 'opp_key', 'is_home', 'date', 'start_ts', 'competition', 'comp_id', 'is_friendly', 'is_competitive', 'status', 'rest_days', 'games_last_7', 'games_last_14', 'is_midweek', 'euro_trip_recent', 'is_post_intl_break']


## 11 · Disponibilidad — bajas por equipo

Una fila por equipo y partido. Columnas: **`event_id`**, **`side`** (home/away) y el recuento de
ausencias: **`n_missing`** (total), **`n_out`** (fuera seguro), **`n_injury`** (lesión),
**`n_suspension`** (sanción) y **`n_doubtful`** (duda). Combinada con el valor por jugador
(sección 9) se convierte en *valor ausente ponderado*.


In [25]:
av = vistazo("team_availability.parquet", titulo="Disponibilidad (bajas)", inventario=False)
print("Todas las columnas:", list(av.columns))


Disponibilidad (bajas)  ->  6,401 filas x 7 columnas


,event_id,side,n_missing,n_out,n_injury,n_suspension,n_doubtful
0,7490532,away,2,2,2,0,0
1,7490532,home,1,1,1,0,0
2,7490533,away,4,3,3,0,1
3,7490533,home,3,2,2,0,1
4,7490534,away,4,4,4,0,0


Todas las columnas: ['event_id', 'side', 'n_missing', 'n_out', 'n_injury', 'n_suspension', 'n_doubtful']


## 12 · Alineaciones y minutos — quién jugó y cuánto

Del mismo endpoint del que salen las bajas cuelga la otra mitad del parte: quién **sí**
jugó y cuántos minutos disputó. Es la única fuente del proyecto que baja al nivel del
futbolista, y hace falta porque el bloque de fatiga cuenta *partidos*, que es un
denominador grueso: un suplente que entra diez minutos y un titular que juega los noventa
suman lo mismo. Los minutos son el denominador que faltaba.

**La distinción que gobierna el bloque.** La alineación de un encuentro y los minutos que
cada jugador disputa en él son hechos **del partido**: describen lo que ocurrió, igual que
los goles o el xG, y usarlos crudos sería fuga aunque el once se publique una hora antes
del saque —margen que, además, no existe cuando la predicción se emite con días de
antelación—. El crudo tiene por tanto el mismo estatuto que la estadística de partido de
Sofascore: es materia prima, nunca variable. Todo lo que se publica se calcula mirando
exclusivamente hacia atrás, sobre partidos **anteriores** del mismo equipo. `xi_continuity`
no mide cuánto rota el equipo en el partido que se va a predecir —eso no se sabe— sino
cuánto rotó en el último que jugó.

De esa asimetría se sigue una propiedad útil: a diferencia de las bajas, que dependen de
que la fuente publique el parte médico y por eso pueden faltar en la jornada por jugar,
este bloque es conocible con días de antelación para cualquier partido futuro, porque solo
necesita el pasado.

**Dos artefactos, dos granularidades.** El extractor publica dos tablas porque sirven a
dos consumidores distintos:

| Artefacto | Granularidad | Contenido | Consumidor |
|---|---|---|---|
| `team_lineups.parquet` | Partido / equipo | `xi_continuity`, `xi_continuity_mean3`, `starters_minutes_7`, `starters_minutes_14`, `minutes_hhi`, `n_players_used` | `reducers.reduce_lineups` |
| `player_minutes.parquet` | Jugador-partido | minutos, titularidad, sello de saque y cuota de minutos de la temporada hasta la fecha | `absences.build_absences` |

El segundo no alimenta ninguna variable por sí mismo: es la materia prima que permite
preguntar, para un ausente concreto y en un instante concreto, qué cuota de los minutos de
su equipo venía disputando. Esa pregunta se responde en el notebook siguiente.

**Universo multicompetición.** Se descargan las alineaciones de todos los partidos
competitivos del calendario de los clubes de LaLiga, no solo los de Liga, por la misma
razón por la que la fatiga se calcula multicompetición: los minutos de un miércoles europeo
son carga real, y contarlos solo en Liga subestimaría de forma sistemática el desgaste de
los cuatro o cinco clubes que juegan en Europa, que son además los más fuertes. Un sesgo
correlacionado con la fuerza del equipo no es ruido: es justo el tipo de error que el modelo
aprendería como señal. El precio es una reconstrucción larga —una petición por partido—,
que se paga una sola vez porque el centinela de partidos consultados hace que las pasadas
semanales solo pidan lo nuevo.


In [ ]:
_ORDEN_LINEUPS = ("python scripts/extractors/lineups.py backfill"
                  "  &&  python scripts/extractors/lineups.py process")

li = vistazo("team_lineups.parquet", opcional=_ORDEN_LINEUPS,
    titulo="Alineaciones (rotacion y carga, equipo-partido de LaLiga)", inventario=False)

pm = vistazo("player_minutes.parquet", opcional=_ORDEN_LINEUPS,
    cols=["event_id", "side", "team_key", "season", "competition",
          "player_key", "is_starter", "minutes_played", "minutes_share"],
    titulo="Minutos por jugador y partido (multicompeticion)", inventario=False)

if li is None or pm is None:
    print("\nLa comprobacion de arranque en frio y la cobertura del bloque quedan sin\n"
          "materia sobre la que medirse. No es un fallo del notebook: es la lectura\n"
          "correcta de un bloque opcional que todavia no se ha volcado, la misma que\n"
          "hara el QA del master en el notebook siguiente.")
else:
    # La propiedad que hace legitimo el bloque: en el PRIMER partido de cada equipo y
    # temporada no hay pasado del que mirar, asi que la continuidad tiene que ser nula.
    primeros = (pm[["team_key", "season", "event_id", "side", "start_ts"]]
                .drop_duplicates()
                .sort_values("start_ts")
                .groupby(["team_key", "season"], as_index=False).head(1))
    arranque = li.merge(primeros[["event_id", "side"]], on=["event_id", "side"],
                        how="inner")
    print(f"primeros partidos de equipo-temporada presentes en el bloque: {len(arranque)}")
    print(f"   de ellos con xi_continuity no nula: "
          f"{int(arranque['xi_continuity'].notna().sum())}   (debe ser 0)")

    # Cobertura de cada magnitud. El hueco NO es un fallo de la fuente: es el arranque
    # en frio de cada temporada, que el preprocesado marcara con su indicador.
    STATS = ["xi_continuity", "xi_continuity_mean3", "starters_minutes_7",
             "starters_minutes_14", "minutes_hhi", "n_players_used"]
    display(li[STATS].notna().mean().rename("cobertura").to_frame().style.format("{:.1%}"))


## 13 · Detalle de tiro — la forma de la ocasión

Understat publica, además del agregado por partido de la sección 5, **cada disparo** del
encuentro: el instante, el resultado, la coordenada desde la que se remató, la jugada que lo
originó y el gol esperado que su modelo le asigna. De ese detalle sale la última fuente del
proyecto.

**Qué añade sobre lo que ya hay.** El dataset ya sabe *cuántos* tiros hizo un equipo
(Sofascore) y *cuánto* xG acumuló (Understat). Lo que ninguna de las dos dice es **cómo se
reparte** ese xG entre los disparos, y esa forma de la distribución es información distinta:
dos equipos con quince remates y un gol esperado de 1,5 pueden haber llegado ahí por caminos
opuestos —quince disparos lejanos y forzados, o cinco ocasiones nítidas y diez remates sin
peligro—, y lo que cabe esperar de ellos la jornada siguiente no es lo mismo. El agregado los
confunde; el detalle los separa.

**Las cuatro magnitudes que se publican**, por equipo y partido:

| Magnitud | Qué mide |
|---|---|
| `shot_xg_mean` | gol esperado del disparo típico: la definición directa de calidad de ocasión |
| `shot_dist_mean` | distancia media al centro de la portería, en metros; única lectura territorial del ataque que no depende del modelo de xG |
| `shot_setpiece_xg_pct` | cuota del gol esperado que nace de balón parado: separa al equipo que fabrica jugando del que depende de la estrategia |
| `shot_xg_max` | gol esperado de la mejor ocasión del partido; junto a la media describe la forma de la distribución |

No se publican **recuentos** —el volumen ya viene de Sofascore, y repetirlo con otra
definición sería la peor clase de duplicado: la que parece información nueva— ni la
proporción de tiros dentro del área, que mide lo mismo que la distancia media pero binarizando
una magnitud continua en un límite sin significado para el modelo de xG.

**Qué se excluye del cálculo.** Los **penaltis**, porque tienen siempre la misma distancia y
un gol esperado próximo a 0,76, de modo que su única aportación sería desplazar las cuatro
magnitudes según cuántos hubo —un hecho puntual, no un rasgo de cómo ataca el equipo; es la
misma decisión que ya separa `npxg` de `xg`—. Y los **goles en propia puerta**, que Understat
inscribe en la lista del equipo que los sufre, de manera que contarlos invertiría el signo de
lo que se quiere medir. Un equipo puede terminar sin ningún remate admisible: las cuatro
magnitudes quedan entonces **nulas, no a cero**, porque cero afirmaría que disparó y lo hizo
sin peligro, cuando lo cierto es que no hay distribución de la que hablar.

**Su estatuto: materia prima, nunca variable.** Estas columnas describen los remates del
partido que se quiere predecir, igual que el marcador o el xG, y usarlas crudas sería
pronosticar el resultado a partir de las ocasiones que lo produjeron. Aquí está la diferencia
con el bloque anterior: la rotación llega ya mirando hacia atrás y por eso pasa tal cual al
conjunto de variables, mientras que la calidad de ocasión solo se convierte en variable
después, a través de las ventanas retrospectivas de `features/form.py`. El notebook 03 lo
recoge, y `features/columns.py` la declara como fuga precisamente para que ninguna vía la deje
entrar en crudo.

**Dos artefactos.** Como en la sección anterior, el extractor publica dos tablas con papeles
distintos:

| Artefacto | Granularidad | Papel |
|---|---|---|
| `understat_shots.parquet` | Un disparo por fila | análisis descriptivo; no entra al máster |
| `team_shot_quality.parquet` | Partido / equipo | lo que consume `reducers.reduce_shot_quality` |

El volcado es largo —una petición por partido, como el de alineaciones— y se paga una sola
vez: el centinela de partidos ya descargados hace que las pasadas semanales pidan solo lo
nuevo.


In [ ]:
_ORDEN_TIROS = ("python scripts/extractors/understat_shots.py backfill"
                "  &&  python scripts/extractors/understat_shots.py process")

sh = vistazo("understat_shots.parquet", opcional=_ORDEN_TIROS,
    cols=["date", "match_key", "side", "minute", "player", "result", "situation",
          "xg", "dist_m", "in_box", "is_penalty", "is_setpiece"],
    titulo="Tiros de Understat (un disparo por fila)", inventario=False)

sq = vistazo("team_shot_quality.parquet", opcional=_ORDEN_TIROS,
    titulo="Calidad de ocasion (equipo-partido)", inventario=False)

if sh is None or sq is None:
    print("\nSin el volcado no hay distribucion de remates que describir. El bloque de\n"
          "calidad de ocasion queda entonces vacio en el master, sus medias moviles\n"
          "nacen sin materia prima y la poda por constancia del preprocesado las retira\n"
          "antes de imputar nada: el modelo se ajusta sin ellas, que es el trato\n"
          "previsto para un bloque opcional ausente.")
else:
    # El criterio de exclusion, comprobado sobre el dato: lo que entra en el agregado
    # son solo los remates en juego abierto o de estrategia, nunca penaltis ni goles
    # en propia puerta.
    util = sh.loc[~(sh["is_penalty"].astype(bool) | sh["is_own_goal"].astype(bool))]
    print(f"tiros descargados: {len(sh):,}   admisibles: {len(util):,}"
          f"   excluidos: {len(sh) - len(util):,}")

    # La consecuencia declarada del criterio: un equipo puede quedarse sin ninguna
    # distribucion de la que hablar, y entonces la fila NO existe (no vale cero).
    lados = 2 * sh["match_key"].nunique()
    print(f"lados equipo-partido con tiros: {len(sq):,} de {lados:,} posibles"
          f"   ({1 - len(sq) / lados:.1%} sin remate admisible)")

    # Forma de las cuatro magnitudes. La media y el maximo del gol esperado deben
    # convivir sin confundirse: si fueran redundantes, una de las dos sobraria.
    METRICAS = ["shot_xg_mean", "shot_dist_mean", "shot_setpiece_xg_pct", "shot_xg_max"]
    display(sq[METRICAS].describe().T[["mean", "50%", "std", "min", "max"]]
            .style.format("{:.3f}"))
    print("\ncorrelacion entre la media y el maximo del gol esperado por tiro: "
          f"{sq['shot_xg_mean'].corr(sq['shot_xg_max']):.2f}")


## 14 · Qué se puede verificar cruzando fuentes, y qué no

Las once tablas anteriores se apoyan en proveedores distintos, pero no todas las
magnitudes están cubiertas por más de uno, y conviene decir dónde está el límite antes de
que alguien lo deduzca de un número raro.

**La estadística de partido descansa en una sola fuente.** El detalle de volumen, duelos,
pases y ratings lo publica únicamente Sofascore. FBref, que sería el candidato natural a
contrastarlo, dejó de publicar dato avanzado a nivel equipo-partido en enero de 2026 y hoy
sirve un marcador básico que ya está en el máster por otra vía; por eso `fbref.py` se
conserva como comprobación cruzada y no alimenta la cadena. La consecuencia es que un error
silencioso de Sofascore —una métrica mal mapeada, un partido con los lados invertidos— no
tiene enfrente una segunda medición de la misma magnitud que lo delate columna a columna.

**Lo que sí admite contraste independiente es el remate.** Understat y Sofascore cuentan
disparos por separado, con proveedores, criterios y capturas distintas, de modo que el
recuento por equipo y partido de uno debe parecerse al del otro. No tiene por qué coincidir
exactamente —cada proveedor decide qué es un disparo, y los bloqueos y los remates de
jugada interrumpida se cuentan de manera distinta—, y esa es justamente la razón de mirar
la **distribución de la diferencia** y no la igualdad: un error de mapeo o un partido
cruzado no produce una discrepancia de uno o dos remates, produce una separación
sistemática o una cola imposible.

El contraste es, por tanto, **parcial y de control de calidad**: cubre el volumen ofensivo
y ninguna de las demás familias, y su resultado no entra en el conjunto de variables.
Cualquier fuente que en el futuro restituyera la redundancia perdida tendría el mismo
papel, y por la misma razón: duplicar en la entrada del modelo una magnitud que ya viaja
en ella añade colinealidad, no información.


In [ ]:
# --- Contraste entre proveedores: el recuento de remates -------------------
# Unica magnitud que dos fuentes independientes miden a la vez. No produce
# ninguna columna: su valor es de control de calidad, no de variable.
if sh is None:
    print("El contraste necesita el detalle de tiro de Understat, que es uno de los\n"
          "dos volcados opcionales y todavia no se ha ejecutado. Sin el, la unica\n"
          "magnitud con dos mediciones independientes se queda sin una de las dos y\n"
          "la comprobacion no puede hacerse; el resto del notebook no depende de ella.")
else:
    _NEC = {"sofa_shots_total_shots_home", "sofa_shots_total_shots_away"}
    if not _NEC <= set(so.columns):
        print("Este volcado de Sofascore no trae el recuento de remates: el contraste\n"
              "entre proveedores no puede hacerse sobre el.")
    else:
        _us = (sh.loc[~sh["is_own_goal"].astype(bool)]
                 .groupby(["match_key", "side"]).size().rename("tiros_understat"))
        _so = (so.set_index("match_key")[sorted(_NEC)]
                 .rename(columns={"sofa_shots_total_shots_home": "home",
                                  "sofa_shots_total_shots_away": "away"})
                 .rename_axis(columns="side").stack().rename("tiros_sofascore"))
        cmp_tiros = pd.concat([_us, _so], axis=1).dropna()
        d = cmp_tiros["tiros_understat"] - cmp_tiros["tiros_sofascore"]

        print(f"lados equipo-partido comparables: {len(cmp_tiros):,}")
        print("correlacion entre los dos recuentos: "
              f"{cmp_tiros['tiros_understat'].corr(cmp_tiros['tiros_sofascore']):.3f}")
        print(f"diferencia media (Understat - Sofascore): {d.mean():+.2f}")
        print(f"diferencia absoluta mediana: {d.abs().median():.1f}   "
              f"dentro de +-2 remates: {(d.abs() <= 2).mean():.1%}")
        print(f"lados con separacion mayor de 6 remates: {(d.abs() > 6).sum():,}")

        # Contraste interno del propio agregado: el xG de los remates admisibles debe
        # reconstruir el npxG que Understat publica a nivel de partido. No es una
        # fuente distinta, es la misma vista con mas detalle, de modo que aqui no se
        # verifica al proveedor sino al criterio de exclusion propio.
        if {"us_home_npxg", "us_away_npxg", "match_key"} <= set(us.columns):
            _ok = sh.loc[~(sh["is_penalty"].astype(bool) | sh["is_own_goal"].astype(bool))]
            _agg = _ok.groupby(["match_key", "side"])["xg"].sum().rename("npxg_remates")
            _pub = (us.set_index("match_key")[["us_home_npxg", "us_away_npxg"]]
                      .rename(columns={"us_home_npxg": "home", "us_away_npxg": "away"})
                      .rename_axis(columns="side").stack().rename("npxg_publicado"))
            _j = pd.concat([_agg, _pub], axis=1).dropna()
            _og = (sh.loc[sh["is_own_goal"].astype(bool)]
                     .groupby(["match_key", "side"]).size()
                     .reindex(_j.index).fillna(0).gt(0))
            err = (_j["npxg_remates"] - _j["npxg_publicado"]).abs()

            print(f"\nreconstruccion del npxG desde los remates (n={len(err):,}):")
            print(f"   lados sin gol en propia: error absoluto medio {err[~_og].mean():.4f}"
                  f"   maximo {err[~_og].max():.3f}")
            if _og.any():
                print(f"   lados con gol en propia: error absoluto medio {err[_og].mean():.4f}"
                      f"   (n={int(_og.sum()):,})")
            print("Las dos cifras han de coincidir salvo por el gol en propia, que Understat")
            print("inscribe en la lista del equipo que lo sufre y que el agregado descarta.")
            print("Un residuo concentrado en esas filas confirma que el criterio de exclusion")
            print("hace lo que dice; un residuo repartido por igual apuntaria a otra cosa y")
            print("obligaria a revisar el agregado antes de usarlo.")


## 15 · Siguiente paso

Con estas fuentes extraídas y persistidas en `data/02_processed/`, la fase de extracción queda
cerrada. El siguiente notebook, **`02_CONSOLIDACION_MASTER.ipynb`**, une todas las tablas por
`match_key` en una única tabla maestra a nivel de partido —incluyendo los dos cruces que pesan una baja: *valor por
jugador × disponibilidad* y *minutos por jugador × disponibilidad*— sobre la que después se construyen las
variables del modelo.
